# SINDYc 1 · Intents, close battles and the leverage pilot (exp06, exploratory)

exp05 could not show what a world model is for: most battles were decided at the start, Blue's
setting never changed within a battle, and the settings were mostly "how sloppily Blue fights".
This notebook sets up the data a model *with control* (SINDYc) needs:

1. Team **intents** as the control input $u$
2. Does the simulator replay exactly? (needed for counterfactuals)
3. **Close battles**: sampling the strength ratio directly
4. **Leverage pilot**: from the same start, does the intent change who wins?
5. What each intent does physically
6. The datasets collected

Everything here is exploratory: collected with `allow_dirty`, in `data/raw/exp06_intents/`.
Code: `envs/intents.py`, `envs/episode_sampling.py` (`CloseBattleSampler`),
`scripts/collect_intents.py`, `scripts/build_intent_states.py`.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sindy_wm.envs.episode import play_episode
from sindy_wm.envs.episode_sampling import CloseBattleSampler
from sindy_wm.envs.intents import INTENTS, IntentSchedule
from sindy_wm.envs.scenarios import load_map_info, make_env, walkable_bounds
from sindy_wm.io import storage
from sindy_wm.models.intent_model import add_controls, add_rates
from sindy_wm.paths import DATA_DIR, RAW_DIR

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
RAW, PROCESSED = RAW_DIR / "exp06_intents", DATA_DIR / "processed" / "exp06_intents"
LABELS = [f"{m}/{t}" for m, t in INTENTS]

## 1 · Intents

An intent is a (movement, target) pair the whole Blue team follows (`IntentPolicy`):

| movement | what units do |
|---|---|
| **advance** | attack what is in range, otherwise walk towards the enemy |
| **hold** | hold the line: don't approach; once any Blue unit has an enemy in range, every unit closes in and fights (as advance) |
| **withdraw** | walk away from any enemy closer than 7 (reach is 5.75), and so don't shoot; stand still once out of danger |

| target | |
|---|---|
| **nearest** | each unit shoots its nearest enemy in range |
| **focus** | the team concentrates fire on one enemy |

Six intents (five distinct: withdrawing units don't shoot, so the target rule doesn't matter).
No random actions: aggression is not a decision.

`IntentSchedule` switches between random intents every 1-4 s, and can **branch**: from a drawn
moment it plays one given intent for 4 s, then advance/nearest to the end. The same seed with
different branch intents gives the same battle up to the branch, and one future per intent.

## 2 · Does the simulator replay exactly?

One seed, played six times with a different branch intent. Up to the branch the snapshots must be
identical; after it they should differ.

In [ ]:
bounds = walkable_bounds(load_map_info("10m_vs_10m"))
SEED = 2


def play(seed, policy):
    s = CloseBattleSampler().sample(seed, bounds)
    env = make_env("10m_vs_10m", step_mul=2, spawn=s.spawn, unit_counts=s.unit_counts)
    return play_episode(env, policy, seed=seed, max_steps=480, start_health=s.start_health)


episodes = {
    intent: play(
        SEED, IntentSchedule(branch_intent=intent, branch_seconds=(1.0, 6.0), branch_hold_seconds=4.0)
    )
    for intent in INTENTS
}
branch_step = round(episodes[INTENTS[0]].controls["branch_at"].iloc[0] / 0.125)
reference = episodes[INTENTS[0]].snapshots
for intent in INTENTS[1:]:
    other = episodes[intent].snapshots
    pd.testing.assert_frame_equal(
        reference[reference["step"] <= branch_step].reset_index(drop=True),
        other[other["step"] <= branch_step].reset_index(drop=True),
    )
print(
    f"seed {SEED}: all six battles identical up to the branch at step {branch_step}",
    f"({branch_step * 0.125:.1f} s)",
)

fig, ax = plt.subplots(figsize=(8, 3.5))
for intent, ep in episodes.items():
    hp = ep.snapshots.groupby(["step", "team"])["hp"].sum().unstack("team") / 450
    label = "/".join(intent) + (" (won)" if ep.battle_won else "")
    ax.plot(hp.index * 0.125, hp["blue"] - hp["red"], label=label)
ax.axvline(branch_step * 0.125, color="k", ls=":", lw=1)
ax.axvline(branch_step * 0.125 + 4, color="k", ls=":", lw=1)
ax.set(
    xlabel="seconds",
    ylabel="health margin, Blue - Red",
    title="One battle, six futures (dotted: branch, and back to advance)",
)
ax.legend(fontsize=8);

**Checked:** the battles are identical up to the branch, so the simulator is deterministic given
the seed and the actions. A counterfactual ("what if Blue had held here?") can be measured, not
only predicted.

## 3 · Close battles

`CloseBattleSampler` draws the **strength ratio** (Blue's units x health over Red's) log-uniformly
from 0.85-1.18 and sets the health fractions to match. exp05 drew force ratio and health
independently.

In [ ]:
def strength(index):
    return index["n_blue"] * index["start_health_blue"] / (index["n_red"] * index["start_health_red"])


exp05 = storage.load_episode_index(RAW_DIR / "exp05_10m_sampled" / "10m_sampled_s2_v1")
exp06 = storage.load_episode_index(RAW / "pilot_advance_nearest")
fig, ax = plt.subplots(figsize=(7, 3))
bins = np.linspace(0.3, 2.5, 60)
ax.hist(strength(exp05), bins, alpha=0.6, density=True, label="exp05")
ax.hist(strength(exp06), bins, alpha=0.6, density=True, label="exp06 (close)")
ax.set(xlabel="strength ratio, Blue / Red", ylabel="density")
ax.legend()
print(
    f"share of battles within ±10% strength: exp05 {np.mean(np.abs(np.log(strength(exp05))) < 0.1):.0%}, "
    f"exp06 {np.mean(np.abs(np.log(strength(exp06))) < 0.1):.0%}"
)

## 4 · Leverage pilot

300 starting states. Each is played six times: one intent for the **first 4 s**, then
advance/nearest. If the intent matters, outcomes from the same start should differ.

In [ ]:
index = {i: storage.load_episode_index(RAW / f"pilot_{i[0]}_{i[1]}").set_index("seed") for i in INTENTS}
won = pd.DataFrame({"/".join(i): x["battle_won"] for i, x in index.items()})
margin = pd.DataFrame({"/".join(i): (x["final_hp_blue"] - x["final_hp_red"]) / 450 for i, x in index.items()})
best_share = margin.idxmax(axis=1).value_counts(normalize=True)
pilot = pd.DataFrame(
    {"win rate": won.mean(), "mean final margin": margin.mean(), "best for (share of states)": best_share}
)
display(pilot.round(3))
print(
    f"best single intent wins {won.mean().max():.1%};",
    f"picking the best of six per state would win {won.max(axis=1).mean():.1%}",
)
decides = (won.max(axis=1) != won.min(axis=1)).mean()
print(f"states where the intent decides the winner (some win, some lose): {decides:.1%}")
base = index[INTENTS[0]]
by_distance = won.groupby(pd.cut(base["spawn_distance"], [0, 6, 9, 12, 17])).mean()
by_distance[["advance/nearest", "hold/nearest", "withdraw/nearest"]].round(2)

**What the numbers say:**

- The intent decides the winner in about 6 of 10 starting states: there is plenty for a
  planner to choose between. Best single intent 60% (hold/nearest), best of six per state 83%.
- **Hold the line is now the best single intent** (60%, against 55% for advance/nearest). The
  first version of hold (units out of range stood still) won 19%: an implementation flaw, fixed
  12 Nov.
- **Waiting pays when the teams start far apart:** from 12-17 apart, advance wins 35%, hold 55%,
  withdraw 56%. Close up, advance and hold are equal (about 61-65%).
- Withdraw first, then attack is still the best choice in the most states (38%), but hold and
  advance share the rest.
- Careful: "best of six" is hindsight. It includes luck. Notebook 3 asks how much of it can be
  predicted.

## 5 · What each intent does physically

From the pilot's state tables: closing speed, damage per unit in reach, and how fast units die
relative to health lost, by intent.

In [ ]:
pilot_states = pd.concat(
    [
        pd.read_parquet(PROCESSED / f"pilot_{m}_{t}.parquet").assign(run=f"{m}/{t}").reset_index()
        for m, t in INTENTS
    ]
)
pilot_states["episode_idx"] = pilot_states["run"] + pilot_states["episode_idx"].astype(str)
pilot_states = add_rates(add_controls(pilot_states.set_index(["episode_idx", "step"])))
rows = pilot_states[(pilot_states["alive_blue"] > 0) & (pilot_states["alive_red"] > 0)]

closing = rows.pivot_table(
    index=pd.cut(rows["distance"], [0, 5, 5.5, 6, 6.5, 7, 8, 9, 10, 20]),
    columns="movement",
    values="d_distance",
    observed=True,
)
display(closing.round(2).rename_axis("distance  \\  d(distance)/dt"))

fire = []
for (m, t), g in rows.groupby(["movement", "target"]):
    ok = g[g["health_red"] > 0.03]
    x = ok["alive_red"] / ok["health_red"] * ok["d_health_red"]
    fire.append(
        {
            "movement": m,
            "target": t,
            "Red's losses per Blue unit in reach": -np.dot(g["reach_blue"], g["d_health_red"])
            / np.dot(g["reach_blue"], g["reach_blue"]),
            "Blue's losses per Red unit in reach": -np.dot(g["reach_red"], g["d_health_blue"])
            / np.dot(g["reach_red"], g["reach_red"]),
            "Red units lost per health lost (p)": np.dot(x, ok["d_alive_red"]) / np.dot(x, x),
        }
    )
pd.DataFrame(fire).set_index(["movement", "target"]).round(3)

**What the numbers say:**

- **Closing speed** far apart: about 6 when Blue advances (both sides walk at about 3), about 3
  when Blue holds or withdraws (only Red walks). In contact a withdrawing Blue opens the distance;
  a holding Blue closes in like an advancing one.
- **Firepower:** a withdrawing Blue deals almost no damage (0.005 against 0.17-0.20). Holding and
  advancing deal the same damage per unit in reach. Red's damage per unit in reach is the same
  whatever Blue does.
- **Focus fire** deals slightly *less* damage per unit in reach, but kills units sooner
  (p about 0.69 against 0.49 in the table): the same health loss removes more firepower.

These are the control effects the SINDYc model in notebook 2 should find by itself.

## 6 · Datasets

In [ ]:
summary = []
for path in sorted(RAW.iterdir()):
    if not (path / "manifest.json").exists():
        continue
    manifest, idx = storage.load_manifest(path), storage.load_episode_index(path)
    summary.append(
        {
            "set": path.name,
            "battles": len(idx),
            "Blue wins": idx["battle_won"].mean(),
            "mean seconds": idx["n_steps"].mean() * 0.125,
            "description": manifest["description"],
        }
    )
pd.set_option("display.max_colwidth", 90)
pd.DataFrame(summary).set_index("set").round(2)

## Questions for you

1. Hold the line only differs from advance before contact: Blue waits instead of walking. Why
   would waiting win more often, especially from far apart?
2. "Best of six wins 80%" is hindsight. What experiment would tell you how much of that a
   planner could ever get?
3. The pilot holds the intent for 4 s. Would you expect more or less leverage at 1 s? At 10 s?